In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor

/Users/viktor/repos/stackoverflow-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
csv_path = Path.cwd().parents[1] / "exports" / "survey_2025_scalar_columns.csv"
df = pd.read_csv(csv_path, low_memory=False)

In [16]:
pd.options.display.max_columns = 500

df

,ResponseId,MainBranch,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,Currency,CompTotal,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,CommPlatformHaveEntr,CommPlatformWantEntr,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,8.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",14.0,"Developer, mobile",20 to 99 employees,People manager,Remote,"Yes, I influenced the purchase of a substantial addition to the tech stack",Work,Fintech,I'm not sure,I have neither consider or transitioned into a new career or industry,7.0,3.0,Ukraine,EUR European Euro,52800.0,Yes,Yes,Yes,No,Yes,NaN,NaN,Yes,Yes,A few times per week,Between 5 and 10 years,I have never participated in Q&A on Stack Overflow,Neutral,"Rarely, almost never","Yes, I use AI tools monthly or infrequently",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,"Yes, I use AI agents at work monthly or infrequently",Not at all or minimally,61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,2.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",10.0,"Developer, back-end",500 to 999 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibility)",No,Personal Project,Retail and Consumer Services,I'm not sure,I have transitioned into a new career and/or industry voluntarily,6.0,5.0,Netherlands,EUR European Euro,90000.0,Yes,Yes,Yes,Yes,Yes,NaN,NaN,Yes,Not sure/can't remember,Multiple times per day,Between 10 and 15 years,"Infrequently, less than once per year","Yes, somewhat",About half of the time,"Yes, I use AI tools weekly",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,"No, and I don't plan to",Not at all or minimally,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-employed",10.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",12.0,"Developer, front-end",NaN,NaN,NaN,No,Work,Software Development,No,I have transitioned into a new career and/or industry involuntarily,3.0,3.0,Ukraine,UAH Ukrainian hryvnia,2214000.0,Yes,Yes,Yes,Yes,Yes,NaN,NaN,Yes,Not sure/can't remember,A few times per week,Between 5 and 10 years,"Infrequently, less than once per year",Neutral,About half of the time,"Yes, I use AI tools daily",Favorable,Somewhat trust,Neither good or bad at handling complex tasks,"Yes, I use AI agents at work weekly","Yes, somewhat",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,4.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",5.0,"Developer, back-end","10,000 or more employees",Individual contributor,Remote,No,Personal Project,Retail and Consumer Services,No,I have neither consider or transitioned into a new career or industry,NaN,NaN,Ukraine,EUR European Euro,31200.0,Yes,No,Yes,Yes,No,NaN,NaN,No,Yes,A few times per month or weekly,Between 3 and 5 years,I have never participated in Q&A on Stack Overflow,Neutral,"Rarely, almost never","Yes, I use AI tools weekly",Favorable,Somewhat tru

In [ ]:
# Auto EDA
# profile = ProfileReport(df, title="Stack Overflow Developer Survey 2025 - Public Results (Scalars)")
# profile.to_file("so_survey_2025_profile_scalars_report.html")

Summarize dataset:  52%|█████▏    | 26/50 [00:01<00:01, 15.08it/s, Describe variable: AIModelsChoice]      /Users/viktor/repos/stackoverflow-project/.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1354: RuntimeWarning: overflow encountered in scalar multiply
  numerator = count * (count + 1) * (count - 1) * m4
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 129.59it/s]


### Baseline with Autogluon (no Feature Engineering): predict `ConvertedCompYearly`

**Goal:** Numbers we can beat later, with almost no feature engineering work.

- **Target:** `ConvertedCompYearly` (USD). Drop rows where it is missing or non-positive.
- **Leakage:** `CompTotal` and `Currency` must be removed. `ConvertedCompYearly` is derived from them. `ResponseId` is just a row ID.
- **Outer split:** `sklearn.model_selection.train_test_split` (80/20 holdout for evaluation).
- **Inside `fit`:** AutoGluon does its own train/validation split on the 80% for model selection; we only hand it clean data plus preprocessing.

In [ ]:
TARGET = "ConvertedCompYearly"
LEAK_COLS = ["ResponseId", "CompTotal", "Currency"]


df = df[df[TARGET].notna() & (df[TARGET] > 0)].reset_index(drop=True)
df = df.drop(columns=[c for c in LEAK_COLS if c in df.columns])

train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True
)

print(f"Rows: {len(df):,}  |  Train: {len(train_data):,}  |  Test: {len(test_data):,}")
train_data.head()

Rows: 23,947  |  Train: 19,157  |  Test: 4,790


,MainBranch,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,CommPlatformHaveEntr,CommPlatformWantEntr,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange,ConvertedCompYearly,JobSat
18879,I am a developer by profession,35-44 years old,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",Employed,3.0,"Yes, I am not new to coding but am learning ne...","Yes, I learned how to use AI-enabled tools req...",5.0,Data engineer,100 to 499 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibi...",No,Work,Software Development,No,I have transitioned into a new career and/or i...,3.0,0.0,United Kingdom of Great Britain and Northern I...,Yes,Yes,Yes,No,Yes,NaN,NaN,Yes,Yes,A few times per month or weekly,Between 3 and 5 years,"Infrequently, less than once per year","No, not really","Rarely, almost never","Yes, I use AI tools daily",Very favorable,Somewhat trust,I don't use AI tools for complex tasks / I don...,"No, I use AI exclusively in copilot/autocomple...",Not at all or minimally,59962.0,10.0
3162,I am a developer by profession,25-34 years old,"Secondary school (e.g. American high school, G...",Employed,6.0,"No, I am not new to coding and did not learn n...","No, I didn't spend time learning in the past year",6.0,"Developer, front-end",20 to 99 employees,Individual contributor,Remote,No,Personal Project,Higher Education,Yes,I have somewhat considered changing my career ...,13.0,5.0,Germany,Yes,Yes,Yes,Yes,Yes,NaN,NaN,NaN,No,A few times per month or weekly,Between 5 and 10 years,I have never participated in Q&A on Stack Over...,"No, not at all","Rarely, almost never","Yes, I use AI tools daily",Favorable,Somewhat distrust,Very poor at handling complex tasks,"Yes, I use AI agents at work monthly or infreq...","Yes, to a great extent",81210.0,8.0
15568,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,13.0,"Yes, I am not new to coding but am learning ne...","Yes, I learned how to use AI-enabled tools for...",19.0,"Developer, back-end","10,000 or more employees",Individual contributor,"Your choice (very flexible, you can come in wh...",No,Work,Fintech,No,I have somewhat considered changing my career ...,10.0,NaN,Romania,Yes,Yes,Yes,No,Yes,NaN,NaN,Yes,Yes,A few times per week,Between 10 and 15 years,Less than once every 2 - 3 months,"Yes, somewhat","Rarely, almost never","Yes, I use AI tools monthly or infrequently",Favorable,Neither trust nor distrust,Neither good or bad at handling complex tasks,"No, I use AI exclusively in copilot/autocomple...","Yes, somewhat",88171.0,8.0
1731,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,19.0,"No, I am not new to coding and did not learn n...","Yes, I learned how to use AI-enabled tools for...",19.0,"Developer, full-stack",Less than 20 employees,Individual contributor,Remote,No,Work,Software Development,I'm not sure,I have somewhat considered changing my career ...,20.0,0.0,United States of America,Yes,Yes,Yes,Yes,Yes,NaN,NaN,No,Yes,A few times per month or weekly,Between 10 and 15 years,I have never participated in Q&A on Stack Over...,"No, not at all","Rarely, almost never","Yes, I use AI tools daily",Favorable,Somewhat trust,"Good, but not great at handling complex tasks","No, but I plan to","Yes, somewhat",121000.0,9.0
16899,I am learning to code,18-24 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,1.0,"Yes, I am not new to coding but am learning ne...","Yes, I learned how to use AI-enabled tools for...",2.0,Data or business analyst,500 to 999 employees,Individual contributor,"Hybrid (some remote, leans heavy to in-person)",No,Personal Project,

In [11]:
predictor = TabularPredictor(
    label=TARGET,
    problem_type="regression",
    eval_metric="mean_absolute_error",
).fit(
    train_data=train_data,
    time_limit=120,
    presets="medium",
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260409_164042"
Preset alias specified: 'medium' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.5
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.3.0: Wed Jan 28 20:54:46 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.93 GB / 32.00 GB (18.5%)
Disk Space Avail:   85.50 GB / 926.35 GB (9.2%)
Presets specified: ['medium']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/Users/viktor/repos/stackoverflow-project/src/notebooks/AutogluonModels/ag-20260409_164042"
Train Da

In [12]:
predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost,-41017.689700,-45790.750140,mean_absolute_error,0.077924,0.023965,54.441336,0.077924,0.023965,54.441336,1,True,4
1,WeightedEnsemble_L2,-42092.222806,-45661.670355,mean_absolute_error,0.187236,0.067031,76.953582,0.002592,0.000672,0.022399,2,True,10
2,NeuralNetTorch,-47150.471113,-51048.327160,mean_absolute_error,0.061188,0.029993,20.234648,0.061188,0.029993,20.234648,1,True,8
3,NeuralNetFastAI,-51330.245656,-56285.635119,mean_absolute_error,0.114567,0.021778,16.579628,0.114567,0.021778,16.579628,1,True,6
4,LightGBMXT,-54143.325229,-57277.274171,mean_absolute_error,0.015189,0.004525,2.680455,0.015189,0.004525,2.680455,1,True,1
5,LightGBM,-55229.362830,-58319.955404,mean_absolute_error,0.011084,0.004644,1.783612,0.011084,0.004644,1.783612,1,True,2
6,LightGBMLarge,-57563.540059,-57924.795482,mean_absolute_error,0.013138,0.005292,6.358534,0.013138,0.005292,6.358534,1,True,9
7,ExtraTreesMSE,-60858.836356,-63423.749454,mean_absolute_error,0.426635,0.066331,3.336656,0.426635,0.066331,3.336656,1,True,5
8,RandomForestMSE,-62655.376308,-63807.869158,mean_absolute_error,0.710112,0.072382,10.250745,0.710112,0.072382,10.250745,1,True,3
9,XGBoost,-64802.133989,-50647.164340,mean_absolute_error,0.045532,0.012401,2.255199,0.045532,0.012401,2.255199,1,True,7


In [13]:
predictor.evaluate(test_data)

{'mean_absolute_error': -42092.22280603779,
 'root_mean_squared_error': np.float64(-236328.554114311),
 'mean_squared_error': -55851185489.760826,
 'r2': 0.01175716993568432,
 'pearsonr': 0.17224825439183225,
 'median_absolute_error': -17528.203125}

In [14]:
predictor.feature_importance(test_data)

Computing feature importance via permutation shuffling for 41 features using 4790 rows with 5 shuffle sets...
	83.83s	= Expected runtime (16.77s per shuffle set)
	20.7s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
Country,23765.706546,266.352754,1.892929e-09,5,24314.130548,23217.282543
WorkExp,3637.190644,181.924445,7.485726e-07,5,4011.775591,3262.605696
OrgSize,1674.715166,111.221485,2.320719e-06,5,1903.721742,1445.708591
DevType,1670.763631,644.196789,2.198188e-03,5,2997.173724,344.353539
Age,863.826036,143.136893,8.724655e-05,5,1158.546876,569.105195
Industry,797.442310,80.824613,1.249204e-05,5,963.861305,631.023316
YearsCode,677.610839,160.400118,3.501942e-04,5,1007.876899,347.344779
Employment,475.720983,34.725323,3.382814e-06,5,547.220903,404.221063
RemoteWork,344.074728,19.414413,1.211228e-06,5,384.049273,304.100182
EdLevel,329.897501,31.986397,1.047375e-05,5,395.757934,264.037068
